# Orion extractor: event typing with a 14B LLM on a Kaggle GPU

Retests LLM event typing (plan, Phases 4 and 7) with models too big for the local 6 GB GPU. Triggers and
participants stay as they are; only the event **type** changes (`llm_types` in `extractor/events.py`):

- `lexicon`: today's verb lexicon (the baseline: triggers_typed 0.195 over the 4 gold stories)
- `<model> all`: the LLM types every trigger
- `<model> other`: the LLM types only the triggers the lexicon left OTHER

Ceiling: 86 gold events have an exactly matching trigger and the lexicon types 69 of them right, so perfect typing
reaches 0.243. Locally qwen2.5:7b scored 0.079 (all) and 0.110 (other) against the lexicon's 0.189 on part 1.
The table also counts non-OTHER events, since the world state keeps every one of them (precision matters there).

**Before running:** Settings → Accelerator **GPU T4 x2**, Settings → **Internet on**, then *Run All* (or *Save &
Run All*). If Kaggle refuses a second GPU session, stop the extractor server notebook first.
About 15 min of setup, then ~5–10 min per variant.

**After running:** download `orion_event_typing.zip` from the Output panel.


In [ ]:
REPO = "https://github.com/SkySwordSaint26/Orion---World-State-Engine.git"
BRANCH = "feat/remote-extractor"            # use "main" once this branch is merged
EVENT_LLMS = ["qwen2.5:14b", "phi4"]        # Ollama library, ~9 GB each at Q4; both fit one T4
MODES = ["all", "other"]
RELATIONS_MODEL = "nuextract2-4b"           # facts/relationships don't affect the event scores; the smallest is fastest
WORK = "/kaggle/working"

## 1. GPU and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess
os.chdir(WORK)
!rm -rf orion && git clone -q --depth 1 -b {BRANCH} {REPO} orion && git -C orion log --oneline -1
EXT = f"{WORK}/orion/extractor"

## 2. Python 3.12 environment

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 {WORK}/venv
PY = f"{WORK}/venv/bin/python"
!uv pip install -q -p {PY} -r {EXT}/requirements.txt
!uv pip install -q -p {PY} --no-deps booknlp==1.0.8
!uv pip install -q -p {PY} https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
# The pinned torch must see the GPU; if Kaggle's driver is too old for its CUDA build, fall back to a CUDA 12.6 build.
check = lambda: subprocess.run([PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
                               capture_output=True, text=True).stdout.strip()
print(check())
if not check().endswith("True"):
    !uv pip install -q -p {PY} --reinstall torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
    print(check())
assert check().endswith("True"), "torch cannot see the GPU: is the accelerator set to GPU T4 x2?"

## 3. Ollama and the models

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
print(urllib.request.urlopen("http://localhost:11434/api/version").read().decode())

In [ ]:
template = open(f"{EXT}/nuextract.Modelfile").read().split("\n", 1)[1]
path = subprocess.run([PY, "-c", "from huggingface_hub import hf_hub_download as d; "
                       "print(d('numind/NuExtract-2.0-4B-GGUF', 'NuExtract-2.0-4B-Q4_K_M.gguf'))"],
                      capture_output=True, text=True, check=True).stdout.strip()
open(f"{WORK}/{RELATIONS_MODEL}.Modelfile", "w").write(f"FROM {path}\n{template}")
subprocess.run(["ollama", "create", RELATIONS_MODEL, "-f", f"{WORK}/{RELATIONS_MODEL}.Modelfile"], check=True,
               capture_output=True)
for model in EVENT_LLMS:
    subprocess.run(["ollama", "pull", model], check=True, capture_output=True)
!ollama list

## 4. Gold evaluation per variant

The first run also downloads GLiNER2 and BookNLP (~3 GB).

In [ ]:
import json, pathlib, time, collections, shutil
OUT = pathlib.Path(f"{WORK}/results"); OUT.mkdir(exist_ok=True)
VARIANTS = {"lexicon": {}} | {f"{m} {mode}": {"ORION_EVENT_LLM": m, "ORION_EVENT_LLM_MODE": mode}
                              for m in EVENT_LLMS for mode in MODES}

def run(args, env):
    p = subprocess.run([PY, "-m", "extractor", *args], cwd=EXT, capture_output=True, text=True,
                       env={**os.environ, "ORION_LLM_MODEL": RELATIONS_MODEL, **env})
    if p.returncode:
        print(p.stdout[-3000:], p.stderr[-3000:]); p.check_returncode()

for name, env in VARIANTS.items():
    folder = OUT / name.replace(" ", "_").replace(":", "-")
    t = time.time()
    run(["predict", "--gold-dir", "data/gold", "--out-dir", str(folder / "gold")], env)
    run(["score", "data/gold", str(folder / "gold"), "--json", str(folder / "gold_scores.json")], env)
    print(f"{name}: {time.time() - t:.0f}s", flush=True)

## 5. Results

In [ ]:
metrics = ["triggers_exact", "triggers_typed", "event_arguments", "event_arguments_entity"]
print(f"{'variant':24}" + "".join(f"{m:>24}" for m in metrics) + f"{'non-OTHER events':>18}")
for name in VARIANTS:
    folder = OUT / name.replace(" ", "_").replace(":", "-")
    micro = json.load(open(folder / "gold_scores.json"))["micro"]
    types = collections.Counter(e["type"] for p in (folder / "gold").glob("*.json") for e in json.load(open(p))["events"])
    print(f"{name:24}" + "".join(f"{micro[m]['f1']:>24.3f}" for m in metrics)
          + f"{sum(types.values()) - types['OTHER']:>18}")
shutil.make_archive(f"{WORK}/orion_event_typing", "zip", OUT)
print("download from the Output panel:", f"{WORK}/orion_event_typing.zip")